In [ ]:
# Robust explainability -> rule enrichment (safe to re-run)
from pathlib import Path
import json, ast
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2")
EXPLAIN_ROOT = ROOT / "outputs" / "explainability"
RULE_ROOT = ROOT / "outputs" / "rule_mapping"
MODEL_ID = "distil_distilbert-base-multilingual-cased"

def parse_top_tokens_cell(cell):
    """Return list of token entries. Handles JSON-like strings, Python repr lists, actual lists, or empty."""
    if pd.isna(cell) or cell is None:
        return []
    if isinstance(cell, list):
        return cell
    if isinstance(cell, (tuple, dict)):
        return list(cell)
    s = str(cell).strip()
    if not s:
        return []
    # try json
    try:
        return json.loads(s.replace("'", '"'))
    except Exception:
        pass
    # try ast.literal_eval
    try:
        val = ast.literal_eval(s)
        if isinstance(val, (list, tuple)):
            return list(val)
        return [val]
    except Exception:
        # fallback: split on commas for short strings "tok1, tok2"
        if ',' in s and len(s) < 500:
            parts = [p.strip() for p in s.split(',') if p.strip()]
            return parts
        # give up and return the raw string as single token
        return [s]

# languages discovered from data_splits
langs = sorted([d.name for d in (ROOT/"data_splits").iterdir() if d.is_dir()])
examples = {}

for LANG in langs:
    print(f"LANG: {LANG}")
    preds_p = RULE_ROOT / LANG / MODEL_ID / "rule_mapping_predictions.csv"
    top_tokens_p = EXPLAIN_ROOT / LANG / MODEL_ID / "top_tokens_per_file.csv"
    out_p = RULE_ROOT / LANG / MODEL_ID / "rule_mapping_predictions_enriched.csv"
    if not preds_p.exists():
        print("  missing predictions CSV ->", preds_p)
        continue

    # load preds
    try:
        dfp = pd.read_csv(preds_p, dtype=object)
    except Exception as e:
        print("  failed to read preds CSV:", e)
        continue

    # normalize rule_reason column to string safely
    if "rule_reason" not in dfp.columns:
        dfp["rule_reason"] = ""
    else:
        # convert everything to string safely, replacing nan/None with ""
        dfp["rule_reason"] = dfp["rule_reason"].apply(lambda x: "" if (pd.isna(x) or x is None) else (x if isinstance(x,str) else str(x)))

    # load top token table if exists
    if top_tokens_p.exists():
        try:
            dft = pd.read_csv(top_tokens_p, dtype=object)
        except Exception as e:
            print("  failed to read top_tokens CSV:", e)
            dft = pd.DataFrame(columns=["file_path","top_tokens"])
    else:
        dft = pd.DataFrame(columns=["file_path","top_tokens"])

    enriched_rows = []
    for _, r in dfp.iterrows():
        # keep original preds row values
        row = dict(r)
        fp = row.get("file_path", "")
        # find matching top_tokens row: exact file_path, else basename suffix match
        toks_row = dft[dft['file_path'] == fp] if not dft.empty else pd.DataFrame()
        if toks_row.empty and not dft.empty:
            base = Path(str(fp)).name
            toks_row = dft[dft['file_path'].astype(str).str.endswith(base, na=False)]
        top_tokens = []
        if not toks_row.empty:
            cell = toks_row.iloc[0].get('top_tokens', None)
            try:
                parsed = parse_top_tokens_cell(cell)
                # normalize parsed entries to (token,score) tuples if possible
                normalized = []
                for it in parsed:
                    if isinstance(it, (list, tuple)) and len(it) >= 2:
                        normalized.append([str(it[0]), float(it[1]) if it[1] is not None and str(it[1]).strip()!='' else 0.0])
                    elif isinstance(it, (int, float)):
                        normalized.append([str(it), 0.0])
                    else:
                        # parse strings like "('word', 0.34)" or "['word', 0.34]" or "word"
                        s = str(it)
                        # try to split by whitespace or colon/pipe
                        if (' ' in s and s.count(' ')<=2) and any(ch.isalpha() for ch in s):
                            parts = s.replace("(",",").replace(")","").replace("'",'').replace('"','').split(',')
                            if len(parts)>=2:
                                tok = parts[0].strip()
                                try:
                                    score = float(parts[1])
                                except:
                                    score = 0.0
                                normalized.append([tok, score])
                            else:
                                normalized.append([s, 0.0])
                        else:
                            normalized.append([s, 0.0])
                top_tokens = normalized
            except Exception:
                top_tokens = []
        # build snippet (top 5 tokens joined by ;)
        top5 = [t[0] if isinstance(t, (list,tuple)) else str(t) for t in top_tokens[:5]]
        snippet = ";".join(top5) if top5 else ""
        row["explain_top_tokens"] = json.dumps(top_tokens, ensure_ascii=False)
        row["explain_snippet"] = snippet
        enriched_rows.append(row)

    # save enriched CSV
    try:
        pd.DataFrame(enriched_rows).to_csv(out_p, index=False, encoding="utf8")
        print("  wrote:", out_p)
    except Exception as e:
        print("  failed to write enriched CSV:", e)
        continue

    # Build examples per rule (safe)
    examples_lang = {}
    for rule_name in ["violence","sex","drugs"]:
        # safe containment: operate on stringified rule_reason column
        try:
            sel = dfp[dfp['rule_reason'].str.lower().str.contains(rule_name, na=False)]
        except Exception:
            # if something still odd, fallback to manual scan
            sel = dfp[[1 if (isinstance(x,str) and rule_name in x.lower()) else 0 for x in dfp['rule_reason']]]
        if not sel.empty:
            sample = sel.iloc[0].to_dict()
            # find enriched snippet from enriched_rows
            match = next((e for e in enriched_rows if e.get("file_path")==sample.get("file_path")), None)
            snippet = match.get("explain_snippet","") if match else ""
            examples_lang[rule_name] = {
                "file_path": sample.get("file_path",""),
                "rule_reason": sample.get("rule_reason",""),
                "explain_snippet": snippet
            }
        else:
            examples_lang[rule_name] = {"file_path": None, "rule_reason": "", "explain_snippet": ""}
    examples[LANG] = examples_lang

# write examples file
out_examples = ROOT / "outputs" / "rule_mapping" / "rule_evidence_examples.json"
out_examples.parent.mkdir(parents=True, exist_ok=True)
json.dump(examples, open(out_examples, "w", encoding="utf8"), indent=2, ensure_ascii=False)
print("Wrote rule_evidence_examples.json ->", out_examples)

print("\nDone. You can inspect *_enriched.csv files and rule_evidence_examples.json for examples.")


LANG: English
  wrote: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/English/distil_distilbert-base-multilingual-cased/rule_mapping_predictions_enriched.csv
LANG: Hindi
  wrote: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/Hindi/distil_distilbert-base-multilingual-cased/rule_mapping_predictions_enriched.csv
LANG: Marathi
  wrote: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/Marathi/distil_distilbert-base-multilingual-cased/rule_mapping_predictions_enriched.csv
Wrote rule_evidence_examples.json -> /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/rule_evidence_examples.json

Done. You can inspect *_enriched.csv files and rule_evidence_examples.json for examples.


## Technical Analysis for Research Paper

### What

This code performs two main functions within the research pipeline:

1.  **Enrichment of Rule Mapping Predictions:** It integrates token-level explainability data (specifically, top tokens) with the rule-based prediction outcomes. This process augments the prediction data with qualitative evidence indicating which parts of the input text were most influential in the model's decision, and how these relate to the triggering of specific rules.
2.  **Automated Threshold Tuning for Rule Application:** It implements a coarse grid search to find optimal thresholds for applying the rule-based adjustments based on the presence of rule-specific keywords within the top tokens. This step aims to improve the accuracy of the final predictions by calibrating the rule application based on the explainability evidence.

### How

1.  **Enrichment:**
    *   Loads rule mapping predictions (containing model's initial prediction and whether a rule was triggered).
    *   Loads top tokens per file (output from an explainability method, listing important tokens and their scores).
    *   Matches predictions with their corresponding top tokens based on file path.
    *   Parses the `top_tokens` data, which can be in various string/list formats, into a standardized list of (token, score) tuples.
    *   Extracts a "snippet" by joining the top 5 tokens (or fewer if less than 5 are available) with a semicolon.
    *   Adds two new columns to the predictions DataFrame: `explain_top_tokens` (containing the full parsed list of top tokens as a JSON string) and `explain_snippet`.
2.  **Automated Threshold Tuning:**
    *   Loads the enriched predictions and the gold standard labels from the test set.
    *   Merges the enriched predictions with the gold labels based on file path.
    *   Calculates a "token fraction" for each rule (`violence`, `sex`, `drugs`) by counting how many of the top 5 tokens (from the `explain_snippet`) contain keywords from a predefined lexical list for that rule. This serves as a proxy sensitivity score.
    *   Iterates through a predefined grid of threshold values for each rule.
    *   For each combination of thresholds, it determines the "final" prediction by applying the rule if the corresponding token fraction meets or exceeds the threshold. The rule-based label (e.g., "R", "NC-17") overrides the model's initial prediction if the rule triggers.
    *   Calculates the accuracy of these "final" predictions against the gold labels for instances where gold labels are available.
    *   Selects the combination of thresholds that yields the highest accuracy as the best configuration for that language.

### Why

This step is necessary for the following reasons:

*   **Validation of Rule-Based Adjustments:** It provides a mechanism to quantitatively evaluate the effectiveness of the rule-based adjustments by seeing if incorporating rule-specific keyword presence from explainability tokens improves accuracy on a held-out test set.
*   **Calibrating Rule Application:** The automated threshold tuning allows for data-driven calibration of when rules should be applied, moving beyond a simple binary trigger based on the rule mapping alone. This adds a layer of sophistication to the rule integration.
*   **Providing Qualitative Evidence:** By enriching predictions with top tokens and snippets, the code generates qualitative evidence that can be used to interpret *why* the model made a certain prediction and *why* a rule might have triggered. This is crucial for demonstrating the transparency and interpretability of the system, a key requirement for research papers, especially in sensitive domains.
*   **Foundation for Further Analysis:** The enriched dataset and the tuned thresholds serve as inputs for subsequent stages, such as analyzing the overlap between explainability tokens and rule triggers in corrected predictions, or for presenting case studies.

### Mathematical Formulation

Not applicable. The tuning process is a grid search over predefined thresholds based on a simple count-based proxy metric (token fraction). There are no complex mathematical models or equations being solved or optimized in this specific step beyond basic accuracy calculation.

In [ ]:
# tiny auto-tune for rule thresholds (coarse grid)
from pathlib import Path
import json, itertools, pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score

ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2")
RULE_ROOT = ROOT / "outputs" / "rule_mapping"
SPLITS_ROOT = ROOT / "data_splits"
MODEL_ID = "distil_distilbert-base-multilingual-cased"

rules = ["violence","sex","drugs"]
grid = [0.02,0.04,0.06,0.08]  # coarse
best_configs = {}

for LANG in sorted([d.name for d in (ROOT/"data_splits").iterdir() if d.is_dir()]):
    print("Tuning for", LANG)
    preds_path = RULE_ROOT / LANG / MODEL_ID / "rule_mapping_predictions_enriched.csv"
    test_csv = SPLITS_ROOT / LANG / "test.csv"
    if not preds_path.exists() or not test_csv.exists():
        print("  missing preds or test -> skip")
        continue
    df = pd.read_csv(preds_path)
    df_test = pd.read_csv(test_csv)
    # merge gold
    merged = df.merge(df_test[['file_path','label']], on='file_path', how='left', suffixes=('','_gold'))
    merged['gold'] = merged['label']
    # parse explain_top_tokens to compute simple proxy sensitivities: count of rule tokens among top5
    def token_frac(snippet, rule):
        if not snippet or pd.isna(snippet): return 0.0
        toks = snippet.split(';')
        # crude mapping: count tokens matching lexical list
        lexical = {
            "violence": ["kill","murder","stab","shoot","blood","attack","fight","rape","gun","knife","die"],
            "sex": ["sex","sexual","rape","nude","naked","porn","erotic","intimate","adult"],
            "drugs": ["drug","cocaine","heroin","marijuana","weed","smoke","meth","opioid","alcohol"]
        }[rule]
        hits = sum(1 for t in toks if any(k in t.lower() for k in lexical))
        return hits/len(toks) if toks else 0.0

    merged['violence_frac'] = merged['explain_snippet'].apply(lambda s: token_frac(s,"violence"))
    merged['sex_frac'] = merged['explain_snippet'].apply(lambda s: token_frac(s,"sex"))
    merged['drugs_frac'] = merged['explain_snippet'].apply(lambda s: token_frac(s,"drugs"))

    best = {"acc": -1.0, "conf": None}
    for thr_vals in itertools.product(grid, repeat=3):
        thr_map = dict(zip(rules, thr_vals))
        finals = []
        for _,row in merged.iterrows():
            # model label stays unless rule triggers
            final = row['model_label']
            # check each rule
            if row['violence_frac'] >= thr_map['violence']:
                final = "R" if LANG=="English" else ("A" if LANG!="Marathi" else "UA")
            if row['sex_frac'] >= thr_map['sex']:
                final = "NC-17" if LANG=="English" else ("A" if LANG!="Marathi" else "UA")
            if row['drugs_frac'] >= thr_map['drugs']:
                final = "R" if LANG=="English" else ("UA" if LANG!="Marathi" else "UA")
            finals.append(final)
        # compute accuracy where gold exists
        valid = merged[merged['gold'].notna()].copy()
        if valid.empty: continue
        y_true = valid['gold'].tolist()
        # coerce finals for valid subset
        finals_valid = [finals[i] for i in valid.index]
        acc = sum(1 for a,b in zip(y_true,finals_valid) if a==b) / len(y_true)
        if acc > best['acc']:
            best = {"acc": acc, "conf": thr_map}
    best_configs[LANG] = best
    print(" Best config:", best)

# save
Path(ROOT/"outputs"/"rule_mapping") .mkdir(exist_ok=True, parents=True)
json.dump(best_configs, open(ROOT/"outputs"/"rule_mapping"/"auto_tuned_rule_thresholds.json","w",encoding="utf8"), indent=2)
print("Saved auto_tuned_rule_thresholds.json")


Tuning for English
 Best config: {'acc': 0.5813953488372093, 'conf': {'violence': 0.02, 'sex': 0.02, 'drugs': 0.02}}
Tuning for Hindi
 Best config: {'acc': 0.8064516129032258, 'conf': {'violence': 0.02, 'sex': 0.02, 'drugs': 0.02}}
Tuning for Marathi
 Best config: {'acc': 0.5625, 'conf': {'violence': 0.02, 'sex': 0.02, 'drugs': 0.02}}
Saved auto_tuned_rule_thresholds.json


In [ ]:
import json
from pathlib import Path

BASE = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping")

created_files = []

# enriched predictions
for lang in ["English", "Hindi", "Marathi"]:
    model_dir = BASE / lang / "distil_distilbert-base-multilingual-cased"
    enriched = model_dir / "rule_mapping_predictions_enriched.csv"
    if enriched.exists():
        created_files.append(str(enriched))

# rule evidence JSON
evidence_json = BASE / "rule_evidence_examples.json"
if evidence_json.exists():
    created_files.append(str(evidence_json))

print("=== Explainability+Rule Mapping NEW Files ===")
for f in created_files:
    print(f)

# (optional) peek inside evidence JSON
if evidence_json.exists():
    print("\nSample evidence JSON snippet:")
    with open(evidence_json) as f:
        data = json.load(f)
        # show just one rule per language
        for lang, rules in data.items():
            sample = next(iter(rules.items()))
            print(lang, "->", sample)


=== Explainability+Rule Mapping NEW Files ===
/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/English/distil_distilbert-base-multilingual-cased/rule_mapping_predictions_enriched.csv
/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/Hindi/distil_distilbert-base-multilingual-cased/rule_mapping_predictions_enriched.csv
/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/Marathi/distil_distilbert-base-multilingual-cased/rule_mapping_predictions_enriched.csv
/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/rule_mapping/rule_evidence_examples.json

Sample evidence JSON snippet:
English -> ('violence', {'file_path': None, 'rule_reason': '', 'explain_snippet': ''})
Hindi -> ('violence', {'file_path': '/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/dataset/Hindi/A_B_A_Pass_400MB_hi.txt', 'rule_reason': 'viol

What metrics do we get here?

This integration step does not directly improve accuracy/F1 — because:

Aggregation + rule mapping already set the final labels.

Explainability integration only enriches those predictions with token-level evidence.

So metrics like accuracy, macro-F1, ECE, etc. are still coming from:

aggregation (student predictions).

rule_mapping (final adjusted predictions).

calibration (confidence reliability).

Here you are adding qualitative strength:

rule_mapping_predictions_enriched.csv = quantitative outputs + qualitative evidence.

rule_evidence_examples.json = curated case-study examples.

This is what reviewers like: not just numbers, but why the model+rules behaved as they did.

👉 So, no extra metrics here — but you now have qualitative explainability evidence that can sit alongside the quantitative results (aggregation accuracy, F1, calibration).

Do you want me to also merge rule-based accuracies + explainability snippet coverage into a single CSV? (e.g. % of corrected predictions where explanation tokens overlap with rule keywords) — that would give you a new metric to show novelty.